# Phase 9: Model Improvement Notebook — MachineMind AI

**Project:** MachineMind AI  
**Dataset:** NASA IMS Bearing Dataset (Subset: Set 2, 984 snapshots)  
**Task:** Unsupervised Vibration Anomaly Detection  
**Phase:** Phase 9 of 11 — Model Improvement  

---

## Notebook Overview
This notebook documents the execution and empirical synthesis of controlled one-factor-at-a-time (OFAT) experiments (`EXP-04` through `EXP-11`) performed during Phase 9 to evaluate candidate model improvements.

### Strict Scoping Rules
- **Fit Partition:** Snapshots `0..159` ($N=160$, ~26.7 hrs). Model fitting and scaler transformations.
- **Calibration Partition:** Snapshots `160..179` ($N=20$, ~3.3 hrs). Threshold derivation ($P_{99}$).
- **Diagnostic Comparison Window:** Snapshots `180..199` ($N=20$, ~3.3 hrs). Comparative model selection.
- **Evaluation Partition:** Snapshots `200..983` ($N=784$, ~130.7 hrs). Held out completely from tuning.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.models import AnomalyDetectionPipeline
from src.evaluation import apply_persistence_filter

print("Environment initialized successfully.")

Environment initialized successfully.


## 1. Dataset & Feature Matrix Verification

In [2]:
features_csv = Path.cwd().parent / "data" / "processed" / "features_set2.csv"
df_features = pd.read_csv(features_csv)
print(f"Loaded features matrix shape: {df_features.shape}")
assert not df_features.isna().any().any(), "NaNs found!"
print("Dataset verified clean (0 NaNs, 0 Infs).")

Loaded features matrix shape: (984, 40)
Dataset verified clean (0 NaNs, 0 Infs).


## 2. Summary of Completed Controlled Experiments (EXP-04 to EXP-11)

| Experiment ID | Tested Factor | Control vs. Candidate | Decision & Status |
|---|---|---|---|
| **EXP-04** | Feature Set | Full 36 vs. Time-Domain 28 | Retain 36-feature set as baseline control; 28-feature set documented |
| **EXP-05** | Feature Scaler | RobustScaler vs. StandardScaler | Retain RobustScaler as baseline control (rank-order invariant) |
| **EXP-06** | PCA Rank | k=1 vs. k=2 vs. k=3 | Accept PCA k=3 components (reduces raw false alarms and calibration MSE) |
| **EXP-07** | iForest Hyperparams | Grid search n_est x max_samp | Retain n_estimators=100, max_samples="auto" for variance stability |
| **EXP-08** | Threshold Strategy | Fit vs. Calibration percentile cutoffs | Retain Calibration P99 percentile line |
| **EXP-09** | Persistence Filter | k=1 vs. k=3 vs. k=5 | Retain k=3 filter (100% false alarm suppression, 20-min delay) |
| **EXP-10** | Fit Window Length | N_fit=100 vs. N_fit=160 | Retain N_fit=160 as primary baseline fit window |
| **EXP-11** | System Architecture | Per-Channel (4 models) vs. Pooled (1 model) | Retain Per-Channel Architecture for spatial bearing localization |

## 3. Execution of Final Improved Pipeline Models

Fits the final per-channel Isolation Forest and PCA (k=3) models strictly on snapshots `0..159` and calculates scores.

In [3]:
fit_end = 160
cal_end = 180
channels = ["ch1", "ch2", "ch3", "ch4"]

full_cols = [c for c in df_features.columns if any(c.startswith(f"{ch}_") for ch in channels)]

final_models = {}
final_thresholds = {}

for ch in channels:
    ch_cols = [c for c in full_cols if c.startswith(f"{ch}_")]
    X_all = df_features[ch_cols].to_numpy(dtype=np.float64)
    X_fit = X_all[:fit_end]
    
    # 1. Isolation Forest Pipeline
    pipe_iforest = AnomalyDetectionPipeline(model_type="iforest", scaler_type="robust", random_state=42)
    pipe_iforest.fit(X_fit)
    scores_iforest = pipe_iforest.compute_anomaly_scores(X_all)
    t_iforest = float(np.percentile(scores_iforest[fit_end:cal_end], 99.0))
    
    # 2. PCA Reconstruction Error Pipeline (k=3)
    pipe_pca = AnomalyDetectionPipeline(model_type="pca", scaler_type="robust", model_params={"n_components": 3}, random_state=42)
    pipe_pca.fit(X_fit)
    scores_pca = pipe_pca.compute_anomaly_scores(X_all)
    t_pca = float(np.percentile(scores_pca[fit_end:cal_end], 99.0))
    
    final_models[ch] = {"iforest": pipe_iforest, "pca": pipe_pca}
    final_thresholds[ch] = {"iforest": t_iforest, "pca": t_pca}
    
    print(f"Channel {ch}: iForest P99 Threshold={t_iforest:.6f}, PCA (k=3) P99 Threshold={t_pca:.6f}")

print("\nFinal pipeline models fit and verified successfully.")

Channel ch1: iForest P99 Threshold=0.492242, PCA (k=3) P99 Threshold=0.424097


Channel ch2: iForest P99 Threshold=0.451808, PCA (k=3) P99 Threshold=0.588735


Channel ch3: iForest P99 Threshold=0.603034, PCA (k=3) P99 Threshold=0.357422


Channel ch4: iForest P99 Threshold=0.529307, PCA (k=3) P99 Threshold=0.569700

Final pipeline models fit and verified successfully.
